## Narrative Position

**Pipeline step:** Feature 구성과 measurement validity (4/6) — **사전 확장 layer · 우리 팀 차별점 layer 2 후반**

**This notebook answers:** *"seed 30 단어만으로 ESG 어휘를 충분히 포착할 수 있는가? fastText 기반 확장 사전을 도입하면 측정이 어떻게 달라지는가? 그리고 그 확장은 cheap-talk 가능성을 *줄이는가* 아니면 오히려 *증폭시키는가*?"*

**Position:** `03_feature_build_v2` 다음. `04_feature_validation_v2`의 입력이 된다.

**Linked robustness evidence:**
- `../jiwoo/02_seed_dictionary_validation.ipynb` — seed별 corpus 등장 빈도와 개별 ρ
- `../전처리/이동원_0519_전처리.ipynb` — manual_v1 expanded dictionary 결과 (참조용 인용)
- `../전처리/김혜성_0519_전처리.ipynb` — 희소 seed 5개 진단 (참조용 인용)

**Evaluation rubric coverage:**
- 평가표 *Feature 종류*: seed TF-IDF + expanded dictionary TF-IDF + fastText 확장 + θ 비교 + 후보 단어 직접 검토
- 평가표 *Feature 해석*: 각 feature가 무엇을 측정하는지 쉬운 말로 설명

**Why this matters for our team's framing:**
사전 확장은 "signal을 더 잘 잡는 도구"로 흔히 소개되지만 — **"무엇을 ESG 단어로 부를 것인가"의 정의 자체를 분석자가 선택**하는 단계다. 이 선택이 결과에 어떻게 작용하는지가 measurement validity의 본질이다. 확장 사전이 ρ를 *높이는* 방향으로만 작동하면 "잘 잡았다"가 아니라 *boilerplate를 더 끌어들였을 가능성*도 점검해야 한다.

> 전체 narrative 흐름은 `00_NARRATIVE_README.md` 참조.

---


# 03b · Expanded Dictionary & fastText Expansion

**Input** : `data/04_preprocessed/tokenized_exp_F.csv` (canonical Kiwi 토큰화)  
**Reference inputs (팀원 환경)** : `seed_dictionary.csv` (30 단어), `expanded_dictionary_manual_v1.csv` (7 단어 추가)  
**Output (현재)** : `data/05_features/seed_sparsity_diagnostics.csv` (재계산 가능)  
**Output (목표)** : `data/05_features/features_expanded_exp_F.parquet` — *현 시점 final/ 파이프라인에서는 미생성*. 본 노트북은 그 gap을 명시적으로 다룬다.

---

## 1. 이 노트북이 다루는 평가표 요구

평가표(`04_submission_and_evaluation.md`)는 fastText 확장에 대해 다음을 명시한다:

> *"expanded dictionary를 seed 10개씩에서 출발해 embedding으로 후보를 넓히고, 팀 기준으로 걸러낸 사전으로 설명했는가?"*  
> *"theta, 후보 단어 직접 검토·걸러내기, 불용어·회사명 제거 기준을 어떻게 정했는가?"*  
> *"fastText 확장 시 cosine threshold θ를 직접 몇 가지 값으로 바꿔가며 결과를 비교한다."*

이 요구를 충족하려면 (a) seed에서 출발해 후보를 넓힌 절차, (b) θ 값을 바꿔본 비교, (c) 후보 직접 검토로 무엇을 살리고 무엇을 버렸는지, 그리고 (d) 그 확장이 *측정에 어떤 차이를 만들었는지*가 모두 보여야 한다.

## 2. 현재 final/ 파이프라인의 상태와 honest gap statement

`03_feature_build_v2`가 산출한 `features_exp_F.parquet`에는 **seed_tfidf_E/S/G** 컬럼만 있고 **expanded_tfidf_E/S/G**는 없다. 즉:

- 현재 main analysis(04 → 05 → 06)는 **seed-only TF-IDF** 위에서 돌아간다.
- expanded TF-IDF는 **`전처리/이동원_0519_전처리.ipynb`의 별도 환경**(29 firm-year subset)에서 manual_v1 사전으로 한 번 계산되었다.
- θ 임계값을 바꿔가며 정량 비교한 표는 **현재 코드베이스에 명시적으로 보이지 않는다.**

우리 팀 narrative는 이 gap을 *숨기지 않고 기록한다*. 그 자체가 measurement fragility의 일부다 — "무엇을 측정으로 부를 것인가"의 선택이 환경마다 달랐다는 사실은 곧 *분석자 의존성*을 의미한다.


## 3. Seed Dictionary 30 — 출발점

교수가 문헌 기반으로 정한 30개 seed (`01_assignment_overview.md` line 134-138).


In [ ]:
# === Seed 30 — 코드 안에서 명시적으로 정의 ===
# 외부 seed_dictionary.csv가 없어도 본 노트북이 self-contained 하도록 인라인 정의.
SEED_30 = {
    'E': ['탄소','온실가스','탄소중립','넷제로','재생에너지','에너지','전력','폐기물','재활용','폐수'],
    'S': ['안전','산업재해','중대재해','임직원','노동','인권','교육훈련','협력사','공급망','지역사회'],
    'G': ['이사회','사외이사','감사위원회','독립성','윤리','준법','컴플라이언스','부패방지','주주','의결권'],
}
import pandas as pd
seed_df = pd.DataFrame([(d, w) for d, ws in SEED_30.items() for w in ws],
                       columns=['dim','seed_word'])
print(f'seed total: {len(seed_df)}  ·  dims: {sorted(SEED_30)}')
seed_df.head()


## 4. Seed 희소도 진단 — 어떤 seed가 corpus에서 사실상 안 쓰이는가

**근거 1 (팀원 분석 인용):** `김혜성_0519_전처리.ipynb` 셀 [54]에서 "희소 seed 5개"를 식별한다 — *넷제로 9회 / 산업재해 ... (상세는 원 노트북 참조)*. 이 희소 seed들이 expanded dictionary 확장의 가장 직접적 근거다.

**근거 2 (본 노트북에서 재계산):** canonical 토큰화 결과(`tokenized_exp_F.csv`) 위에서 seed별 corpus 등장 빈도와 *고유 등장 문서 수(document frequency)*를 다시 측정한다. 이 두 지표가 함께 낮으면 "이 seed는 우리 표본에서 ESG 어휘로 쓰이지 않는다"는 판단 근거가 된다.


In [ ]:
# === seed sparsity 재계산 — tokenized_exp_F.csv 위에서 ===
import os, json
from pathlib import Path
import pandas as pd
from collections import Counter

PROJ = Path.cwd()
while PROJ.name and not (PROJ/'data').exists() and PROJ.parent != PROJ:
    PROJ = PROJ.parent

tok_path = PROJ/'data'/'04_preprocessed'/'tokenized_exp_F.csv'
if not tok_path.exists():
    raise FileNotFoundError(f'{tok_path} 없음 — 02_preprocess_kiwi_v2를 먼저 실행')

tok = pd.read_csv(tok_path)
# 토큰 컬럼은 공백으로 join된 string으로 저장되어 있다고 가정
token_col = next((c for c in ['tokens','tokenized_text','token_string'] if c in tok.columns), None)
if token_col is None:
    # 안전 fallback: 첫 string 컬럼
    token_col = next(c for c in tok.columns if tok[c].dtype == object)
print(f'토큰 컬럼: {token_col} · 문서 수: {len(tok)}')

SEED_30_FLAT = {w: d for d, ws in SEED_30.items() for w in ws}

doc_freq = Counter()
term_freq = Counter()
for s in tok[token_col].fillna('').astype(str):
    toks = s.split()
    present = set()
    for t in toks:
        if t in SEED_30_FLAT:
            term_freq[t] += 1
            present.add(t)
    for t in present:
        doc_freq[t] += 1

sparsity = pd.DataFrame({
    'seed_word': list(SEED_30_FLAT),
    'dim': [SEED_30_FLAT[w] for w in SEED_30_FLAT],
    'term_freq': [term_freq.get(w, 0) for w in SEED_30_FLAT],
    'doc_freq':  [doc_freq.get(w, 0)  for w in SEED_30_FLAT],
}).sort_values(['dim','doc_freq']).reset_index(drop=True)
# 희소 판정: doc_freq <= 10 (전체 ~213 문서 대비 ~5%)
sparsity['is_sparse'] = sparsity['doc_freq'] <= 10
print('\n희소 seed (doc_freq ≤ 10):')
print(sparsity.loc[sparsity['is_sparse']].to_string(index=False))

out_dir = PROJ/'data'/'05_features'
out_dir.mkdir(parents=True, exist_ok=True)
sparsity.to_csv(out_dir/'seed_sparsity_diagnostics.csv', index=False, encoding='utf-8-sig')
print(f'\n저장: {out_dir/"seed_sparsity_diagnostics.csv"}')


### Interpretation — 희소 seed의 의미

`doc_freq`가 낮은 seed는 *우리 corpus의 ESG 어휘로 거의 기능하지 않는다*. 두 가지 해석이 가능하다:

1. **"한국 사업보고서에서 이 개념은 실제로 다른 표현으로 쓰인다"** — 예: "넷제로" 대신 "탄소중립"이 일반적. 이 경우 expanded dictionary로 대안 표현을 끌어와야 측정 타당도가 올라간다.
2. **"이 개념 자체가 한국 사업보고서에서 잘 나타나지 않는다"** — 그러면 확장도 도움이 안 된다. 정직하게 "이 ESG 차원은 본 corpus에서 약하게 포착된다"고 보고한다.

두 해석은 사후적으로 구분된다 — fastText 후보를 보고서 직접 결정해야 한다.


## 5. fastText 후보 확장 — 절차와 θ 비교 frame

**원칙 (Bao et al., 2024 인용 — 평가표 line 102):** seed 단어 주변 코사인 유사도가 일정 임계값 θ 이상인 단어를 후보로 추리고, 그 후보를 *직접 읽고* 도메인 관점에서 채택/기각한다.

**절차 (4 step):**
1. 사업보고서 corpus로 학습된 fastText embedding 위에서, 각 seed에 대해 cosine top-K (예: K=100) 후보를 뽑는다.
2. cosine threshold θ를 [0.50, 0.60, 0.70, 0.80] 등 몇 가지 값으로 바꿔가며 후보 수와 잡음 비율을 비교한다.
3. **상위 100 단어를 직접 훑어** 회사명·일반어·boilerplate를 제거한다.
4. 남은 후보를 도메인 분류표(ENV/SOC/GOV)에 매핑한다.

**현 시점 코드베이스 상태:**
- 본 단계의 **재계산용 fastText 모델 자체가 final/ 코드베이스 안에 없다.** 학습은 팀원(이동원/신지영) 환경에서 별도 수행되었고, 결과(7개 단어)만 manual_v1 사전으로 압축되어 있다.
- 따라서 θ 비교표는 *현재 노트북에서 생성*되지 않는다. 대신 아래에 frame과 *목표 표 양식*을 제시하고, 재계산 시 채워 넣을 수 있도록 둔다.


### θ 비교 — Frame 표 (재계산 시 채워 넣는다)

| θ | 후보 단어 수(누적) | 직접 검토 후 채택 수 | 잡음 비율 | 비고 |
|---:|---:|---:|---:|---|
| 0.50 |  |  |  | 후보 매우 많음, 잡음 증가 우려 |
| 0.60 |  |  |  | |
| 0.70 |  |  |  | 비교적 보수적 |
| 0.80 |  |  |  | 보수적, 누락 가능성 |

**평가표 정합:** 위 표를 채우고 **최종 채택 θ와 그 이유**를 Decision Box(아래)에 기록한다.

### 후보 검토 절차 — 어떤 단어를 살리고 어떤 단어를 버리는가

- **채택 기준:** ESG 도메인에서 *실제로 사용되는 표현*이며, seed와 의미적으로 정합. 예: "탄소" → "이산화탄소", "CO2", "GHG".
- **기각 기준:** (a) 회사명·고유명사, (b) 일반 비즈니스 용어(매출, 영업이익), (c) 너무 일반적("기업", "활동"), (d) 이미 seed에 포함된 어형 변종.
- **불용어·회사명 제거:** Kiwi 사용자 사전(02_preprocess_kiwi_v2)에서 처리된 부분 외에, 후보 검토 단계에서 한 번 더 거른다.


## 6. Manual_v1 Expanded Dictionary — 팀원 결과 인용

이동원님 환경(`전처리/이동원_0519_전처리.ipynb` cell #41)에서 다음이 확인된다:

- **seed v1: 30 단어** (E/S/G 각 10)
- **expanded_manual v1: 7 단어** (수동 검토 후 채택)
- TF-IDF shape: (29 firm-year, 3,288 vocab) — `min_df=2, max_df=0.95`
- seed TF-IDF + expanded TF-IDF + cosine similarity 세 가지 feature가 차원별(E/S/G)로 계산됨

**한계 (honest gap):**
- 본 결과는 *batch 30 firm-year 중 SUCCESS 28건* 위에서 산출되었다. canonical final/ 파이프라인의 N=210과 같지 않다.
- expanded 단어 *7개의 구성*은 manual_v1.csv 파일을 직접 봐야 확인되는데, 그 파일은 현재 connected folder에 동기화되어 있지 않다.
- θ 비교 결과가 manual_v1과 어떻게 연결되는지(어떤 θ에서 이 7개가 선택되었는지)는 원 notebook output에서 확인되지 않는다.

이 한계들은 *우리 통합 분석의 측정 fragility*를 보여주는 또 다른 layer다 — 같은 corpus에서 같은 팀 안에서도 *어떤 환경에서 누가 계산했는가*에 따라 표본·사전·θ가 달라졌다.


## 7. Measurement Validity — seed-only vs expanded 비교

평가표 *측정 validity* 항목은 "feature가 KCGS 등급과 통계적으로 연관되는가"를 묻는다. 그 검증은 `04_feature_validation_v2`에서 수행된다.

확장 사전이 도입되었을 때 예상되는 패턴은 *세 가지*다:

1. **"확장이 도움이 된다"**: ρ가 일관되게 (절대값) 커지고, 부호가 안정. 그러면 seed-only는 단순히 *희소 표현 누락* 때문에 약했다.
2. **"확장이 cheap-talk를 끌어들인다"**: ρ의 절대값은 비슷하거나 줄어들고, 후보 단어가 verbosity와 강하게 공선성을 보인다. 그러면 확장은 ESG 신호가 아니라 *분량 신호*를 강화한 것.
3. **"부호가 뒤집힌다"**: 확장 후 ρ의 부호가 바뀐다면, 이는 우리가 정의한 ESG 어휘의 외연이 *KCGS의 외부 평가와 정합하지 않는다*는 measurement validity 문제로 직결된다.

**우리 팀 기대치 (가설):** manual_v1의 7개 단어는 도메인 검토로 거른 결과라 1번 패턴에 가까울 가능성이 있지만, **verbosity와 강한 공선성을 보이면 Alpha 1의 verbosity-adjusted score와 함께 읽어야 한다.** 즉 expanded TF-IDF의 의미는 04 검증과 06 alpha를 거쳐야 비로소 결정된다.


## 8. Decision Box — Expanded Dictionary 선택

- **Alternative:**
  - (A) seed-only TF-IDF (현재 final/ 파이프라인)
  - (B) fastText θ별 자동 확장 (검토 없이 전부 채택)
  - (C) fastText 후보 → **수동 검토 후 채택** (manual_v1)
  - (D) 외부 ESG 사전 차용 (예: 기존 ESG 키워드 리스트)

- **Choice:** 현 시점 우리 메인 분석은 **(A) seed-only**이고, **(C) manual_v1**은 팀원 환경의 보완 결과로 인용한다.

- **Justification:**
  - (B)는 평가표가 명시적으로 경계하는 "검토 없이 ESG 사전으로 확정" 케이스라 채택하지 않는다.
  - (D)는 외부 사전 자체가 boilerplate를 포함할 수 있고, 우리 corpus 특성을 반영하지 않는다.
  - (A)는 단순하지만 *희소 seed* 문제를 그대로 받는다.
  - (C)는 도메인 검토를 통과한 단어만 추가하므로 측정 타당도와 reproducibility의 균형이 좋다.

- **Limitation:**
  - manual_v1은 7 단어로 한정되어 확장 효과가 작을 수 있다.
  - θ 비교 표가 본 노트북에서 *완전히 채워지지 않은 상태*다 — 이 자체가 measurement fragility의 직접 증거다.
  - 확장 사전이 KCGS 등급과 더 강하게 연관된다고 해서 그것이 "진짜 ESG signal"임을 보증하지 않는다 — verbosity와 boilerplate 공선성 점검이 04/06 단계에서 필수다.


## 9. 평가표 체크포인트 — 이 노트북이 커버한 항목

- [x] seed 30개 정의를 코드 안에 명시
- [x] seed 희소도 진단 (재계산 가능)
- [x] expanded dictionary 절차 4 step 명시 + Bao et al. 인용
- [x] θ 비교 frame 표 (재계산용 placeholder + 채택 기준 설명)
- [x] 후보 단어 직접 검토 기준 (채택/기각 4 기준)
- [x] 불용어·회사명 제거 위치 (02 + 후보 검토 단계)
- [x] manual_v1 결과 인용 + N, vocab 등 정량 정보 기록
- [x] honest gap statement (현재 final/는 seed-only, expanded는 팀원 환경)
- [x] Decision Box (A/B/C/D 대안 + Choice + Justification + Limitation)
- [x] cheap-talk 가능성 frame (1/2/3 패턴 해석)
- [ ] θ 정량 비교 표 채우기 — fastText 재학습 환경 확보 후 (다음 cycle)
- [ ] expanded TF-IDF의 04/06 단계 검증 — 본 노트북 output을 입력으로 받는 후속 task

## 10. 다음 단계 연결

본 노트북의 결과(seed 희소도, 확장 frame, manual_v1 인용)는 다음 두 단계로 흘러간다:

- `04_feature_validation_v2` — seed-only TF-IDF의 Spearman/MWU 검증을 "희소 seed가 얼마나 신호를 약화하는가"의 관점에서 다시 읽는다.
- `06_alpha_analysis_v2` Alpha 1 — verbosity-adjusted 비교에서, 확장 사전이 도입되면 verbosity 공선성이 어떻게 변하는지가 후속 알파의 후보가 된다.
